# CP-SAT 的基本 API ― 写约束用的工具一览

## 本章要点

CP-SAT 常用 API 的清单，以及每个 API 的运行示例。

- 变量的创建方法（整数、布尔、离散区间）
- 线性约束和带条件的约束（`only_enforce_if`）
- 最大、最小、绝对值，乘、除、取余
- 逻辑约束（AND / OR / 恰好一个 / XOR）
- 允许、禁止的取值组合，数组元素引用（`add_element`），逆映射（`add_inverse`）

In [1]:
from ortools.sat.python import cp_model
from helper.ortools import get_all_solutions, is_between, is_not_between

## 本章涉及的函数

把 CP-SAT 常用的 API 和能运行的例子放在一起，像词典一样方便查阅。每个例子都用 `get_all_solutions()` 取出**所有解**，亲眼确认约束在做什么。

| 分类 | 函数 |
|---|---|
| **创建变量** | `new_int_var`, `new_bool_var`, `new_int_var_from_domain` |
| **目标函数** | `minimize`, `maximize` |
| **线性约束** | `add`, `add_linear_constraint`, `add_linear_expression_in_domain` |
| **最大、最小、绝对值** | `add_min_equality`, `add_max_equality`, `add_abs_equality` |
| **乘、除、取余** | `add_modulo_equality`, `add_multiplication_equality`, `add_division_equality` |
| **互不相同的值** | `add_all_different` |
| **逻辑约束** | `add_implication`, `add_bool_or`, `add_at_least_one`, `add_at_most_one`, `add_exactly_one`, `add_bool_and`, `add_bool_xor` |
| **取值组合** | `add_allowed_assignments`, `add_forbidden_assignments` |
| **数组、元素** | `add_element`, `add_inverse` |

## 创建变量

- `new_int_var`: 指定最小值、最大值、变量名，创建整数变量
- `new_bool_var`: 创建最小值 0、最大值 1 的整数变量。参数只指定变量名
- `new_int_var_from_domain`: 指定不连续的取值范围，创建整数变量

下面用示例说明这几个函数。用辅助函数 `get_all_solutions()` 取出所有解，观察变量能取哪些值。

接下来，创建 $0 \le x \le 3$ 的整数变量。

In [2]:
model = cp_model.CpModel()
x = model.new_int_var(0, 3, 'x')
get_all_solutions(model)

[{'x': 0}, {'x': 1}, {'x': 2}, {'x': 3}]

接下来，创建 $0 \le b \le 1$ 的整数变量（布尔变量）。

In [3]:
model = cp_model.CpModel()
x = model.new_bool_var('b')
get_all_solutions(model)

[{'b': 0}, {'b': 1}]

接下来，用 `Domain.from_values()` 直接指定候选值。

In [4]:
model = cp_model.CpModel()
domain = cp_model.Domain.from_values([1, 3, 5, 7])
x = model.new_int_var_from_domain(domain, 'x')
get_all_solutions(model)

[{'x': 1}, {'x': 3}, {'x': 5}, {'x': 7}]

接下来，用 `Domain.from_flat_intervals()` 指定多个范围。每个范围用两个连续的值（下限和上限）来指定。没有下限或上限时，使用 `INT_MIN` 和 `INT_MAX`。

In [5]:
model = cp_model.CpModel()
domain = cp_model.Domain.from_flat_intervals([1, 3, 7, 9, 10, 12])
x = model.new_int_var_from_domain(domain, 'x')
get_all_solutions(model)

[{'x': 1},
 {'x': 2},
 {'x': 3},
 {'x': 7},
 {'x': 8},
 {'x': 9},
 {'x': 10},
 {'x': 11},
 {'x': 12}]

## 线性约束

- `add()`: 把线性表达式的比较式作为约束条件添加。
- `add_linear_constraint()`: 指定线性表达式的下限和上限，添加约束条件。
- `add_linear_expression_in_domain()`: 把线性表达式限制在不连续的范围里。

下面两个例子都创建 $30 \le 10 x \le 50$ 这个线性约束，不过例 1 用两个 `add()` 分别指定式子的下限和上限。例 2 用 `add_linear_constraint()` 同时设定下限和上限。

In [6]:
model = cp_model.CpModel()
x = model.new_int_var(0, 100, 'x')
model.add(10 * x <= 50)
model.add(10 * x >= 30)
get_all_solutions(model)

[{'x': 3}, {'x': 4}, {'x': 5}]

In [7]:
model = cp_model.CpModel()
x = model.new_int_var(0, 100, 'x')
model.add_linear_constraint(10 * x, 30, 50)
get_all_solutions(model)

[{'x': 3}, {'x': 4}, {'x': 5}]

接下来用 `add_linear_expression_in_domain()`，创建 $30 \leq 10x \leq 50$ **或者** $70 \leq 10x \leq 80$ 的约束。`Domain.from_flat_intervals([30, 50, 70, 80])` 表示“30～50”和“70～80”两个区间。

In [8]:
model = cp_model.CpModel()
x = model.new_int_var(0, 100, 'x')
domain = cp_model.Domain.from_flat_intervals([30, 50, 70, 80])
model.add_linear_expression_in_domain(10 * x, domain)
get_all_solutions(model)

[{'x': 3}, {'x': 4}, {'x': 5}, {'x': 7}, {'x': 8}]

本书还提供下面两个辅助函数。它们生成比较式，可以直接用 `add()` 添加为约束条件。

- $l \le x \le u$: `is_between(x, l, u)`
- $x < l \lor x > u$: `is_not_between(x, l, u)`

In [9]:
model = cp_model.CpModel()
x = model.new_int_var(1, 6, 'x')
y = model.new_int_var(1, 6, 'y')
model.add(is_between(x + y, 9, 10))
get_all_solutions(model)

[{'x': 3, 'y': 6},
 {'x': 4, 'y': 5},
 {'x': 4, 'y': 6},
 {'x': 5, 'y': 4},
 {'x': 5, 'y': 5},
 {'x': 6, 'y': 3},
 {'x': 6, 'y': 4}]

In [10]:
model = cp_model.CpModel()
x = model.new_int_var(1, 6, 'x')
y = model.new_int_var(1, 6, 'y')
model.add(is_not_between(x + y, 4, 11))
get_all_solutions(model)

[{'x': 1, 'y': 1}, {'x': 2, 'y': 1}, {'x': 1, 'y': 2}, {'x': 6, 'y': 6}]

## 条件分支

对创建好的约束条件使用 `only_enforce_if()` 方法，可以指定该条件生效的前提。前提只能是一个或多个布尔变量，或者它们的否定。例如要实现下面这样的条件，可以用一个中间布尔变量把两个比较式连接起来。

$$
1 \leq x \leq 3, \quad 1 \leq y \leq 3
$$
$$
z =
\begin{cases}
50, & \text{if } x + y \geq 4 \\
-50, & \text{otherwise}
\end{cases}
$$

下面的代码用 `only_enforce_if(cond)` 和 `only_enforce_if(~cond)`，把 `cond` 连接到 `x + y >= 4`。然后用这个条件约束 `z` 的值。

In [11]:
model = cp_model.CpModel()
x, y = [model.new_int_var(1, 3, name) for name in 'xy']
z= model.new_int_var(-100, 100, 'z')

cond = model.new_bool_var('cond')
model.add(x + y >= 4).only_enforce_if(cond)
model.add(x + y < 4).only_enforce_if(~cond)

model.add(z == 50).only_enforce_if(cond)
model.add(z == -50).only_enforce_if(~cond)
get_all_solutions(model)

[{'x': 1, 'y': 1, 'z': -50, 'cond': 0},
 {'x': 2, 'y': 1, 'z': -50, 'cond': 0},
 {'x': 1, 'y': 2, 'z': -50, 'cond': 0},
 {'x': 1, 'y': 3, 'z': 50, 'cond': 1},
 {'x': 2, 'y': 2, 'z': 50, 'cond': 1},
 {'x': 3, 'y': 2, 'z': 50, 'cond': 1},
 {'x': 3, 'y': 3, 'z': 50, 'cond': 1},
 {'x': 2, 'y': 3, 'z': 50, 'cond': 1},
 {'x': 3, 'y': 1, 'z': 50, 'cond': 1}]

上面的方法需要手动写出比较式的否定，比较麻烦。本书提供辅助函数 `equals(cond, expr, model)`，它会自动生成 `expr` 的否定，把下面两个约束添加到 `model`。

```python
model.add(expr).only_enforce_if(cond)
model.add(inverse_expr).only_enforce_if(~cond)
```

下面的代码用 `equals()` 把 `x + y >= 4` 这个条件关联到 `cond`。

In [12]:
from helper.ortools import equals
model = cp_model.CpModel()
x, y = [model.new_int_var(1, 3, name) for name in 'xy']
z= model.new_int_var(-100, 100, 'z')

cond = model.new_bool_var('cond')
equals(cond, x + y >= 4, model)

model.add(z == 50).only_enforce_if(cond)
model.add(z == -50).only_enforce_if(~cond)
get_all_solutions(model)

[{'x': 1, 'y': 1, 'z': -50, 'cond': 0},
 {'x': 2, 'y': 1, 'z': -50, 'cond': 0},
 {'x': 1, 'y': 2, 'z': -50, 'cond': 0},
 {'x': 1, 'y': 3, 'z': 50, 'cond': 1},
 {'x': 2, 'y': 2, 'z': 50, 'cond': 1},
 {'x': 3, 'y': 2, 'z': 50, 'cond': 1},
 {'x': 3, 'y': 3, 'z': 50, 'cond': 1},
 {'x': 2, 'y': 3, 'z': 50, 'cond': 1},
 {'x': 3, 'y': 1, 'z': 50, 'cond': 1}]

下面的代码约束 `2 * x` 和 `y` 中较小的那个等于 2。例如 `x=1, y=3` 时 `min(2, 3) = 2`，所以是解。

In [13]:
model = cp_model.CpModel()
x, y = [model.new_int_var(0, 4, n) for n in 'xy']
model.add_min_equality(2, [2 * x,  y])
get_all_solutions(model)

[{'x': 1, 'y': 2},
 {'x': 1, 'y': 3},
 {'x': 1, 'y': 4},
 {'x': 2, 'y': 2},
 {'x': 3, 'y': 2},
 {'x': 4, 'y': 2}]

下面的代码约束 `2 * x` 和 `y` 中较大的那个等于 4。例如 `x=2, y=3` 时 `max(4, 3) = 4`，这样的组合就是解。

In [14]:
model = cp_model.CpModel()
x, y = [model.new_int_var(1, 4, n) for n in 'xy']
model.add_max_equality(4, [2 * x, y])
get_all_solutions(model)

[{'x': 1, 'y': 4},
 {'x': 2, 'y': 1},
 {'x': 2, 'y': 2},
 {'x': 2, 'y': 3},
 {'x': 2, 'y': 4}]

下面的代码约束 `x - y` 的绝对值等于 `2`。例如 `x=3, y=1` 时 `|3 - 1| = 2`，这样的组合就是解。

In [15]:
model = cp_model.CpModel()
x, y = [model.new_int_var(1, 4, n) for n in 'xy']
model.add_abs_equality(2, x - y)
get_all_solutions(model)

[{'x': 1, 'y': 3}, {'x': 2, 'y': 4}, {'x': 3, 'y': 1}, {'x': 4, 'y': 2}]

## 变量之间的乘法和除法

接下来介绍几个方便的方法，用它们可以简单地建模整数变量之间的乘法和除法。

* `add_modulo_equality(target, dividend, divisor)`: 添加 `dividend % divisor == target` 这个约束。
* `add_multiplication_equality(target, *expressions)`: 添加 `product(expressions) == target` 这个约束。
* `add_division_equality(target, numerator, denominator)`: 添加 `numerator // denominator == target` 这个约束（整数除法）。

下面的代码用 `add_multiplication_equality()` 求三边长都是整数的直角三角形（勾股数）。

In [16]:
model = cp_model.CpModel()
max_n = 20
x, y, z = [model.new_int_var(1, max_n, n) for n in 'xyz']
x2, y2, z2 = [model.new_int_var(1, 2 * max_n**2, n) for n in ['x2', 'y2', 'z2']]
model.add_multiplication_equality(x2, x, x)
model.add_multiplication_equality(y2, y, y)
model.add_multiplication_equality(z2, z, z)
model.add(x2 + y2 == z2)
model.add(y >= x)
get_all_solutions(model)

[{'x': 3, 'y': 4, 'z': 5, 'x2': 9, 'y2': 16, 'z2': 25},
 {'x': 5, 'y': 12, 'z': 13, 'x2': 25, 'y2': 144, 'z2': 169},
 {'x': 6, 'y': 8, 'z': 10, 'x2': 36, 'y2': 64, 'z2': 100},
 {'x': 8, 'y': 15, 'z': 17, 'x2': 64, 'y2': 225, 'z2': 289},
 {'x': 9, 'y': 12, 'z': 15, 'x2': 81, 'y2': 144, 'z2': 225},
 {'x': 12, 'y': 16, 'z': 20, 'x2': 144, 'y2': 256, 'z2': 400}]

## 互不相同的值

要让多个变量取互不相同的值，可以用方便的方法 `add_all_different(*expressions)`。它添加约束，让指定的所有表达式**取不同的值**。这在数独、分配问题等需要互不相同值的问题里经常用到。下面的代码求 `1, 2, 3` 的排列。

In [17]:
model = cp_model.CpModel()
x, y, z = [model.new_int_var(1, 3, n) for n in 'xyz']
model.add_all_different(x, y, z)
get_all_solutions(model)

[{'x': 3, 'y': 2, 'z': 1},
 {'x': 3, 'y': 1, 'z': 2},
 {'x': 2, 'y': 3, 'z': 1},
 {'x': 2, 'y': 1, 'z': 3},
 {'x': 1, 'y': 3, 'z': 2},
 {'x': 1, 'y': 2, 'z': 3}]

## 逻辑约束

这一节的约束只接受**布尔变量及其否定**（注意不能直接放整数变量）。

| 方法 | 内容 |
|----------|------|
| `add_implication(a, b)` | **a ⇒ b**（a 为真则 b 也为真） |
| `add_bool_or([a, b, c])` | **a OR b OR c**（至少 1 个为真） |
| `add_at_least_one([a, b, c])` | 至少 1 个为真 |
| `add_at_most_one([a, b, c])` | 最多 1 个为真 |
| `add_exactly_one([a, b, c])` | 恰好 1 个为真 |
| `add_bool_and([a, b, c])` | **a AND b AND c**（全部为真） |
| `add_bool_xor([a, b, c])` | 为真的变量个数是奇数 |

`add_implication(a, b)` 添加逻辑蕴含的约束条件。意思是“如果 a 为真，那么 b 也为真”。这个约束可以用下面的真值表表示。

| 输入 a | 输入 b | 输出 a → b |
|--------|--------|------------|
|   0    |   0    |     1      |
|   0    |   1    |     1      |
|   1    |   0    |     0      |
|   1    |   1    |     1      |

只有当 `a` 为真时 `b` 也为真，“输出 a → b”才是 1。下面的代码求“输出 a → b”为 1 的解，输出 3 个解。

In [18]:
model = cp_model.CpModel()
a, b = [model.new_bool_var(n) for n in 'ab']
model.add_implication(a, b)
get_all_solutions(model)

[{'a': 0, 'b': 0}, {'a': 0, 'b': 1}, {'a': 1, 'b': 1}]

`add_bool_or()` 和 `add_at_least_one()` 意思一样，对布尔变量列表约束至少 1 个为真。也就是说，排除全部为假的组合。

In [19]:
model = cp_model.CpModel()
x, y, z = [model.new_bool_var(n) for n in 'xyz']
model.add_bool_or(x, y, z)
get_all_solutions(model)

[{'x': 0, 'y': 1, 'z': 0},
 {'x': 0, 'y': 1, 'z': 1},
 {'x': 0, 'y': 0, 'z': 1},
 {'x': 1, 'y': 0, 'z': 1},
 {'x': 1, 'y': 0, 'z': 0},
 {'x': 1, 'y': 1, 'z': 0},
 {'x': 1, 'y': 1, 'z': 1}]

`add_at_most_one()` 约束布尔变量列表中最多只有 1 个为真。也就是说，不会有 2 个以上同时为真。

In [20]:
model = cp_model.CpModel()
x, y, z = [model.new_bool_var(n) for n in 'xyz']
model.add_at_most_one(x, y, z)
get_all_solutions(model)

[{'x': 0, 'y': 0, 'z': 0},
 {'x': 0, 'y': 1, 'z': 0},
 {'x': 0, 'y': 0, 'z': 1},
 {'x': 1, 'y': 0, 'z': 0}]

`add_exactly_one()` 约束布尔变量列表中恰好只有 1 个为真。

In [21]:
model = cp_model.CpModel()
x, y, z = [model.new_bool_var(n) for n in 'xyz']
model.add_exactly_one(x, y, z)
get_all_solutions(model)

[{'x': 0, 'y': 1, 'z': 0}, {'x': 0, 'y': 0, 'z': 1}, {'x': 1, 'y': 0, 'z': 0}]

`add_bool_and()` 约束布尔变量列表全部为真。

In [22]:
model = cp_model.CpModel()
x, y, z = [model.new_bool_var(n) for n in 'xyz']
model.add_bool_and(x, y, z)
get_all_solutions(model)

[{'x': 1, 'y': 1, 'z': 1}]

这个约束往往太强，一般和 `only_enforce_if()` 一起使用。

In [23]:
model = cp_model.CpModel()
x, y, z = [model.new_bool_var(n) for n in 'xyz']
model.add_bool_and(x, y).only_enforce_if(~z)
get_all_solutions(model)

[{'x': 1, 'y': 1, 'z': 0},
 {'x': 1, 'y': 1, 'z': 1},
 {'x': 1, 'y': 0, 'z': 1},
 {'x': 0, 'y': 0, 'z': 1},
 {'x': 0, 'y': 1, 'z': 1}]

`add_bool_xor()` 约束布尔变量列表中为真的个数是奇数。

In [24]:
model = cp_model.CpModel()
x, y, z = [model.new_bool_var(n) for n in 'xyz']
model.add_bool_xor(x, y, z)
get_all_solutions(model)

[{'x': 0, 'y': 1, 'z': 0},
 {'x': 0, 'y': 0, 'z': 1},
 {'x': 1, 'y': 0, 'z': 0},
 {'x': 1, 'y': 1, 'z': 1}]

## 取值组合的解

`add_allowed_assignments()` 把指定的变量允许取的取值组合添加为约束。这个约束强制列表中的各个变量只能取特定组合里的值。

下面的代码对变量 `x` 和 `y` 设定特定的组合（`[5, 6]` 和 `[6, 7]`），对变量 `y` 和 `z` 也设定另一组组合（`[6, 8]` 和 `[7, 9]`），然后求允许的解。

In [25]:
model = cp_model.CpModel()
x, y, z = [model.new_int_var(1, 10, n) for n in 'xyz']
model.add_allowed_assignments([x, y], [[5, 6], [6, 7]])
model.add_allowed_assignments([y, z], [[6, 8], [7, 9]])
get_all_solutions(model)

[{'x': 5, 'y': 6, 'z': 8}, {'x': 6, 'y': 7, 'z': 9}]

对多个计算式设定候选解时，用中间变量。下面的代码把 `x_plus_y` 和 `y_plus_z` 定义为中间变量，分别保存 `x + y` 和 `y + z` 的计算式。再用 `add_allowed_assignments()` 允许 `x_plus_y` 为 `5` 或 `6`、`y_plus_z` 为 `6` 或 `7` 的组合。这样借助中间变量，就能简单地给复杂的计算式设定候选解。

In [26]:
model = cp_model.CpModel()
x, y, z = [model.new_int_var(1, 10, n) for n in 'xyz']
x_plus_y = model.new_int_var(-100, 100, 'x + y')
y_plus_z = model.new_int_var(-100, 100, 'y + z')
model.add(x_plus_y == x + y)
model.add(y_plus_z == y + z)
model.add_allowed_assignments([x_plus_y, y_plus_z], [[5, 6], [6, 7]])
get_all_solutions(model)

[{'x': 1, 'y': 4, 'z': 2, 'x + y': 5, 'y + z': 6},
 {'x': 2, 'y': 3, 'z': 3, 'x + y': 5, 'y + z': 6},
 {'x': 3, 'y': 2, 'z': 4, 'x + y': 5, 'y + z': 6},
 {'x': 4, 'y': 1, 'z': 5, 'x + y': 5, 'y + z': 6},
 {'x': 4, 'y': 2, 'z': 5, 'x + y': 6, 'y + z': 7},
 {'x': 3, 'y': 3, 'z': 4, 'x + y': 6, 'y + z': 7},
 {'x': 2, 'y': 4, 'z': 3, 'x + y': 6, 'y + z': 7},
 {'x': 5, 'y': 1, 'z': 6, 'x + y': 6, 'y + z': 7},
 {'x': 1, 'y': 5, 'z': 2, 'x + y': 6, 'y + z': 7}]

`add_forbidden_assignments()` 把指定的变量禁止取的取值组合添加为约束。用这个约束，可以让特定变量的组合不出现在解里。下面的代码对 `x` 和 `y`，以及 `y` 和 `z` 的组合，禁止 `1, 2`、`1, 3`、`2, 2` 这几种组合。

In [27]:
model = cp_model.CpModel()
x, y, z = [model.new_int_var(1, 3, n) for n in 'xyz']
model.add_forbidden_assignments([x, y], [[1, 2], [1, 3], [2, 2]])
model.add_forbidden_assignments([y, z], [[1, 2], [1, 3], [2, 2]])
get_all_solutions(model)

[{'x': 3, 'y': 3, 'z': 1},
 {'x': 3, 'y': 2, 'z': 1},
 {'x': 3, 'y': 1, 'z': 1},
 {'x': 2, 'y': 1, 'z': 1},
 {'x': 2, 'y': 3, 'z': 1},
 {'x': 2, 'y': 3, 'z': 2},
 {'x': 2, 'y': 3, 'z': 3},
 {'x': 3, 'y': 3, 'z': 3},
 {'x': 3, 'y': 2, 'z': 3},
 {'x': 3, 'y': 3, 'z': 2},
 {'x': 1, 'y': 1, 'z': 1}]

## 数组元素

`add_element(index, values, element)` 添加约束：数组 `values` 中第 `index` 个元素等于 `element`。用这个约束，可以设定条件让数组里指定位置的元素等于特定的值。

In [28]:
model = cp_model.CpModel()
index = model.new_int_var(0, 4, 'index')
element = model.new_int_var(-100, 100, 'element')
model.add_element(index, [1, 3, 5, 7], element)
get_all_solutions(model)

[{'index': 0, 'element': 1},
 {'index': 1, 'element': 3},
 {'index': 2, 'element': 5},
 {'index': 3, 'element': 7}]

`values` 不只是常量列表，也可以用变量列表。

In [29]:
model = cp_model.CpModel()
x, y, z =  [model.new_bool_var(n) for n in 'xyz']
index = model.new_int_var(0, 3, 'index')
element = model.new_int_var(-100, 100, 'element')
model.add_at_most_one([x, y, z])
model.add_element(index, [x, y, z], element)
get_all_solutions(model)

[{'x': 0, 'y': 0, 'z': 0, 'index': 0, 'element': 0},
 {'x': 0, 'y': 1, 'z': 0, 'index': 0, 'element': 0},
 {'x': 0, 'y': 0, 'z': 1, 'index': 0, 'element': 0},
 {'x': 0, 'y': 0, 'z': 1, 'index': 1, 'element': 0},
 {'x': 0, 'y': 0, 'z': 0, 'index': 1, 'element': 0},
 {'x': 0, 'y': 1, 'z': 0, 'index': 1, 'element': 1},
 {'x': 0, 'y': 0, 'z': 1, 'index': 2, 'element': 1},
 {'x': 0, 'y': 0, 'z': 0, 'index': 2, 'element': 0},
 {'x': 0, 'y': 1, 'z': 0, 'index': 2, 'element': 0},
 {'x': 1, 'y': 0, 'z': 0, 'index': 2, 'element': 0},
 {'x': 1, 'y': 0, 'z': 0, 'index': 1, 'element': 0},
 {'x': 1, 'y': 0, 'z': 0, 'index': 0, 'element': 1}]

`add_inverse(a, b)` 添加约束：用数组 `a` 的索引决定数组 `b` 的元素。具体来说，约束 `b[a[i]] == i`。这保证 `b` 与 `a` 的 `arg_sort()` 一致。

例如给定 `a` 的元素 `[1, 2, 0]`，`b` 就按 `[2, 0, 1]` 的顺序放置索引，于是 `b[1] == 0`, `b[2] == 1`, `b[0] == 2`。

In [30]:
import numpy as np

model = cp_model.CpModel()
a = [model.new_int_var(0, 2, n) for n in ['a0', 'a1', 'a2']]
b = [model.new_int_var(0, 2, n) for n in ['b0', 'b1', 'b2']]
model.add_inverse(a, b)
for sol in get_all_solutions(model):
    a_sol = [sol[f'a{i}'] for i in range(3)]
    b_sol = [sol[f'b{i}'] for i in range(3)]
    print(f'a = {a_sol}, b = {b_sol}, arg_sort = {np.argsort(a_sol).tolist()}')

a = [2, 1, 0], b = [2, 1, 0], arg_sort = [2, 1, 0]
a = [1, 2, 0], b = [2, 0, 1], arg_sort = [2, 0, 1]
a = [2, 0, 1], b = [1, 2, 0], arg_sort = [1, 2, 0]
a = [1, 0, 2], b = [1, 0, 2], arg_sort = [1, 0, 2]
a = [0, 2, 1], b = [0, 2, 1], arg_sort = [0, 2, 1]
a = [0, 1, 2], b = [0, 1, 2], arg_sort = [0, 1, 2]


## 小结

| 想做的事 | 用的东西 |
|---|---|
| 整数变量 | `new_int_var(lb, ub, name)` |
| 0/1 变量 | `new_bool_var(name)` |
| 只用离散的值 | `new_int_var_from_domain(Domain.from_values([...]), name)` |
| 区间的集合 | `Domain.from_flat_intervals([lb1, ub1, lb2, ub2])` |
| 上下限一起设 | `add_linear_constraint(式, lb, ub)` |
| 收进不连续的范围 | `add_linear_expression_in_domain(式, domain)` |
| 有条件地让约束生效 | `model.add(...).only_enforce_if(b)` |
| 最大、最小、绝对值 | `add_max_equality` / `add_min_equality` / `add_abs_equality` |
| 积、商、余数 | `add_multiplication_equality` / `add_division_equality` / `add_modulo_equality` |
| 全部不同 | `add_all_different` |
| 允许、禁止组合 | `add_allowed_assignments` / `add_forbidden_assignments` |
| 引用数组 | `add_element(index, values, element)` |
| 逆映射 | `add_inverse(a, b)`（`b[a[i]] == i`） |

- 约束就是“不断减少解的候选”。用 `get_all_solutions()` 看解的集合，就能很好理解约束的含义
- 逻辑约束只适用于布尔变量。想用整数时，先把比较式赋给布尔变量再用